# Основные эксперименты: LandCover.ai semantic segmentation

Этот ноутбук запускает один контролируемый experiment run. Код моделей, данных и метрик хранится в репозитории; здесь остаются только параметры Kaggle и orchestration.

Не запускай test-ячейку до фиксации конфигурации и выбора лучшей эпохи по validation foreground mIoU.

In [ ]:
# Kaggle: Settings -> Internet = On. Репозиторий должен быть доступен для clone.
!pip install -q segmentation-models-pytorch albumentations
!git clone --depth 1 https://github.com/abbos-trnv/landcover-segmentation.git /kaggle/working/landcover-segmentation-code
%cd /kaggle/working/landcover-segmentation-code
!git rev-parse HEAD

In [ ]:
from pathlib import Path
import json
import subprocess

import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.data import LandCoverTileDataset
from src.index import build_tile_index
from src.models import create_model
from src.run_utils import create_run_dir, load_config, save_config, save_json, set_seed
from src.train import evaluate, train_one_epoch
from src.transforms import training_transform, validation_transform

PROJECT_DIR = Path('/kaggle/working/landcover-segmentation-code')
DATA_ROOT = Path('/kaggle/input/datasets/adrianboguszewski/landcoverai')
IMAGE_DIR = DATA_ROOT / 'images'
MASK_DIR = DATA_ROOT / 'masks'
CONFIG_PATH = PROJECT_DIR / 'configs/unet_resnet18_scratch.yaml'
RUNS_ROOT = Path('/kaggle/working/runs')

config = load_config(CONFIG_PATH)
set_seed(config['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
code_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()

with open(PROJECT_DIR / 'configs/scene_split.json', encoding='utf-8') as file:
    scene_split = json.load(file)

tile_index = build_tile_index(IMAGE_DIR, scene_split, config['tile_size'])
assert set(tile_index['split']) == {'train', 'val', 'test'}

run_dir = create_run_dir(RUNS_ROOT, config['experiment_name'])
tile_index.to_csv(run_dir / 'tile_index.csv', index=False)
save_config(config, run_dir)
save_json({'code_revision': code_revision, 'device': str(device)}, run_dir / 'environment.json')

print('Code revision:', code_revision)
print('Device:', device)
print(tile_index.groupby('split').size())
print('Run directory:', run_dir)

In [ ]:
# Создание одинаковых DataLoader и модели для одного architecture run.
train_index = tile_index.query("split == 'train'").reset_index(drop=True)
val_index = tile_index.query("split == 'val'").reset_index(drop=True)
test_index = tile_index.query("split == 'test'").reset_index(drop=True)

train_dataset = LandCoverTileDataset(train_index, IMAGE_DIR, MASK_DIR, config['tile_size'], training_transform())
val_dataset = LandCoverTileDataset(val_index, IMAGE_DIR, MASK_DIR, config['tile_size'], validation_transform())
test_dataset = LandCoverTileDataset(test_index, IMAGE_DIR, MASK_DIR, config['tile_size'], validation_transform())

loader_kwargs = dict(batch_size=config['batch_size'], num_workers=2, pin_memory=True)
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

model = create_model(
    config['architecture'], config['encoder_name'], config['encoder_weights'], config['num_classes']
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
save_json({'parameters': parameter_count, 'parameters_m': parameter_count / 1e6}, run_dir / 'model.json')
print(f"Model: {config['architecture']} + {config['encoder_name']} | {parameter_count / 1e6:.2f} M params")

In [ ]:
# Основное обучение. Лучший checkpoint выбирается ТОЛЬКО по validation foreground mIoU.
history = []
best_val_miou = float('-inf')
epochs_without_improvement = 0

for epoch in range(1, config['max_epochs'] + 1):
    train_scores = train_one_epoch(model, train_loader, optimizer, device, config['num_classes'], config['ignore_index'])
    val_scores, _ = evaluate(model, val_loader, device, config['num_classes'], config['ignore_index'])
    scheduler.step(val_scores['foreground_miou'])

    row = {'epoch': epoch, 'learning_rate': optimizer.param_groups[0]['lr']}
    row.update({f'train_{key}': value for key, value in train_scores.items()})
    row.update({f'val_{key}': value for key, value in val_scores.items()})
    history.append(row)
    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)

    improved = val_scores['foreground_miou'] > best_val_miou
    if improved:
        best_val_miou = val_scores['foreground_miou']
        epochs_without_improvement = 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'config': config}, run_dir / 'best_checkpoint.pth')
    else:
        epochs_without_improvement += 1

    print(f"Epoch {epoch:02d} | train loss={train_scores['loss']:.4f} | val fg-mIoU={val_scores['foreground_miou']:.4f} | best={best_val_miou:.4f}")

    if epochs_without_improvement >= config['early_stopping_patience']:
        print('Early stopping: validation foreground mIoU has not improved.')
        break

save_json({'best_val_foreground_miou': best_val_miou, 'epochs_completed': len(history)}, run_dir / 'summary.json')

## Test evaluation — запускать один раз

Запускай следующую ячейку только после того, как конфигурация и лучшая эпоха зафиксированы по validation. Не используй test для выбора learning rate, loss или числа эпох.

In [ ]:
checkpoint = torch.load(run_dir / 'best_checkpoint.pth', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
test_scores, per_scene = evaluate(model, test_loader, device, config['num_classes'], config['ignore_index'])

pd.DataFrame([test_scores]).to_csv(run_dir / 'test_metrics.csv', index=False)
pd.DataFrame(per_scene).to_csv(run_dir / 'test_per_scene_metrics.csv', index=False)
print(pd.Series(test_scores).sort_index())
print(pd.DataFrame(per_scene)[['scene', 'foreground_miou']])